In [ ]:
import os
import ast

def min_max_scaler(signal):
  sig_max = np.max(signal)
  sig_min = np.min(signal)
  return (signal - sig_min) / (sig_max - sig_min + 1e-6)

def right_pad_or_crop(signal, lens, target_len=100, pad_value=0.0):
    """
    Right-pad or right-crop a 1D signal to target_len.
    """
    np_len = np.array(lens)
    cumsum_len = np.cumsum(np_len)
    signal = np.asarray(signal)
    padded_signal = []
    mask = []
    for idx in range(len(cumsum_len)):
      if idx == 0:
        start_index = 0
      else:
        start_index = cumsum_len[idx - 1]
      end_index = cumsum_len[idx]
      segment = signal[start_index:end_index]
      if len(segment) > target_len:
        padded_signal.extend(list(signal[start_index:target_len + start_index]))
        mask.extend([1] * target_len)
      else:
        padded_segment = np.pad(segment, (0, target_len - len(segment)), mode="constant", constant_values=pad_value)
        padded_signal.extend(list(padded_segment))
        mask.extend([1] * len(segment) + [0] * (target_len - len(segment)))
    return np.asarray(padded_signal), np.asarray(mask)


def concat_pulses_from_bin(row, base_dir, fs=100):
    """
    Loads the full waveform once, extracts each valid pulse slice, concatenates them.

    Assumes filename pattern like:
      case_{caseid}_time_{glucose_time}_cleaned_filtered.npy
    """
    lens = []
    caseid = int(row["caseid"])
    glucose_time = int(float(row["glucose_time"]))

    path = os.path.join(base_dir, f"case_{caseid}_time_{glucose_time}_cleaned_filtered.npy")
    full_signal = np.load(path)

    starts = row["pulse_starts"]
    ends = row["pulse_ends"]
    starts = ast.literal_eval(starts)
    ends = ast.literal_eval(ends)
    pulses = []
    for s, e in zip(starts, ends):
        s = int(s); e = int(e)
        if e > s and s >= 0 and e <= len(full_signal):
            pulses.append(full_signal[s:e+1])
            lens.append(e - s+1)

    if len(pulses) == 0:
        return None

    return np.concatenate(pulses, axis=0), lens

In [ ]:
train_meta, val_meta, test_meta = pd.read_csv('100Hz_train_15pulses_demographics.csv'), pd.read_csv('100Hz_val_15pulses_demographics.csv'), pd.read_csv( '100Hz_test_15pulses_demographics.csv')

scaler = StandardScaler()
cont_cols = ['age', 'weight', 'bmi', 'height','actual_hr']
train_meta[cont_cols] = scaler.fit_transform(train_meta[cont_cols])
val_meta[cont_cols] = scaler.transform(val_meta[cont_cols])
test_meta[cont_cols] = scaler.transform(test_meta[cont_cols])

In [ ]:
import tqdm
import numpy as np
import os
import pandas as pd

# Change input path to your directory
INPUT_PATH = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Model/100Hz/demographics_15_bin_concat'
# Base directory which contains filtered signal
base_dir = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/100Hz_4min_windows/BW_spline/'
# Change output path to your directory
OUTPUT_PATH = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Model Data/100Hz_15pulses/bin_concat/test_min_max_norm/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

splits = ['train', 'val', 'test']
demo_cols = ['age','height','weight','bmi', 'actual_hr','sex','preop_dm','preop_htn']

for split in splits:
  X = []
  y = []
  masks = []
  D = []
  if split == 'train':
    df = train_meta
  elif split == 'val':
    df = val_meta
  else:
    df = test_meta
  for idx, row in tqdm.tqdm(df.iterrows(), total=len(df)):
    mask = []
    caseid = int(row['caseid'])
    glucose_time = int(float(row['glucose_time']))
    glucose_value = int(row['glucose_value'])
    win_id = int(row['win_id'])
    input, lens = concat_pulses_from_bin(row, base_dir)
    input = min_max_scaler(input)
    input, mask = right_pad_or_crop(input, lens)
    X.append(input)
    y.append(glucose_value)
    masks.append(mask)
    demo = row[demo_cols].to_numpy(dtype=np.float32)  # shape (F,)
    print(row[demo_cols])
    D.append(demo)

  print(f"X {split} len: {len(X)}")
  print(f"y {split} len: {len(y)}")
  print(f"{split}: X {len(X)}, D {len(D)}, mask {len(masks)}, y {len(y)}")

  np.save(os.path.join(OUTPUT_PATH, f'X_{split}_pad_before_concat.npy'), np.array(X, dtype=np.float32))
  np.save(os.path.join(OUTPUT_PATH, f'y_{split}_pad_before_concat.npy'), np.array(y, dtype=np.float32))
  np.save(os.path.join(OUTPUT_PATH, f'mask_{split}_pad_before_concat.npy'), np.array(masks, dtype=np.int32))
  np.save(os.path.join(OUTPUT_PATH, f'demo_{split}_pad_before_concat.npy'), np.array(D, dtype=np.float32))